# Part 3 — Toy training: UResNet + PPN on the generic detector

You will train the **first network of the SPINE chain**: UResNet (labels every voxel as
shower/track/Michel/delta/low-energy) plus PPN (finds start and end points). It runs for 200
iterations: long enough to watch it learn, short enough to finish within a Jupyter session.
Then you will:
- plot the training and validation **loss curves**;
- **evaluate** every saved checkpoint;
- **plug your weights into the full chain**.

**Session needed:** Jupyter on `ampere` with **1 GPU**. Read sections 1–3 of [`README.md`](README.md) first.

> **Terminal equivalent** (from `compute_shell.sh`):
> ```bash
> RUN=$WORKDIR/experiments/uresnet_ppn/toy
> spine_container.sh spine -c $SPINE_TUTORIAL/configs/toy_train_uresnet_ppn.yaml \
>     -s $GENERIC_TRAIN -n 3200 --val-source $GENERIC_TEST --val-num-entries 64 \
>     --weight-prefix $RUN/weights/snapshot --log-dir $RUN
> ```

## 0. Setup

In [ ]:
import sys; sys.path.insert(0, "..")
import os, glob, shutil
from pathlib import Path
import pandas as pd
import tutorial_env as te
env = te.setup()

if not te.has_gpu():
    print("⚠️  No GPU visible: training will not work here. Restart Jupyter on 'ampere' with 1 GPU.")

TRAIN_FILE = env["GENERIC_TRAIN"]     # on S3DF: the official generic training set
TEST_FILE = env["GENERIC_TEST"]       # ...and its test set (never used for training)
if not te.ON_S3DF:                    # off-site: the tiny public file stands in for both
    TRAIN_FILE = TEST_FILE = str(te.fetch("generic_small.root"))

EXPERIMENTS = Path(env["WORKDIR"]) / "experiments" / "uresnet_ppn"
RUN = EXPERIMENTS / "toy"             # everything from this training goes here
print("training data :", TRAIN_FILE)
print("validation    :", TEST_FILE)
print("run directory :", RUN)

## 1. What will be trained?

The tutorial config [`../configs/toy_train_uresnet_ppn.yaml`](../configs/toy_train_uresnet_ppn.yaml)
includes the official recipe and changes a few numbers. Here is the part that controls the training itself:

In [ ]:
te.sh("spine-config dump $SPINE_TUTORIAL/configs/toy_train_uresnet_ppn.yaml "
      "| grep -E -A4 '^(base|train|validation):'")

- `iterations: 200` with `batch_size: 16`, so the network sees 3,200 events once.
- `save_step: 50` saves a checkpoint every 50 iterations.
- `optimizer: Adam, lr: 0.001` is how big each nudge to the weights is.
- `validation` evaluates on unseen events at every checkpoint, and keeps the best one.

## 2. Train!

A new training refuses to overwrite an old one, so the next cell first moves any previous toy run aside.
Then it starts training. Each line of the output is one iteration; watch the `loss` go down.

In [ ]:
if RUN.exists():
    old = RUN.with_name(RUN.name + "_old")
    shutil.rmtree(old, ignore_errors=True)
    RUN.rename(old)
    print("previous run moved to", old)

te.sh(f"""spine -c $SPINE_TUTORIAL/configs/toy_train_uresnet_ppn.yaml \
          -s {TRAIN_FILE} -n 3200 \
          --val-source {TEST_FILE} --val-num-entries 64 \
          --weight-prefix {RUN}/weights/snapshot \
          --log-dir {RUN}""")

**What you should see:** per-iteration lines with loss and accuracy, a validation summary
every 50 iterations, and no `Traceback` at the end. Then the run directory contains:

In [ ]:
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.relative_to(RUN)}")

- `weights/snapshot-49.ckpt … snapshot-199.ckpt`: checkpoints. Numbers count from 0, so `-49` was saved after 50 iterations.
- `weights/snapshot-best.ckpt`: the checkpoint with the lowest validation loss.
- `train_log-0000000.csv`: one row per iteration. `validation_log-*.csv`: one file per checkpoint.

## 3. Loss curves

The training log is an ordinary CSV:

In [ ]:
train_log = pd.read_csv(RUN / "train_log-0000000.csv")
print(len(train_log), "rows; columns:", list(train_log.columns)[:25], "...")
train_log[["iter", "epoch", "loss"] + [c for c in train_log.columns if "accuracy" in c][:3]].tail()

`TrainDrawer` plots training (thin, noisy) and validation (points) together. It takes the
folder that *contains* the run (`EXPERIMENTS`) and the run's name (`"toy"`). `smoothing`
averages neighbouring iterations to make the trend visible.

In [ ]:
from spine.vis.drawer.train import TrainDrawer

drawer = TrainDrawer(str(EXPERIMENTS), interactive=False, val_prefix="validation")
drawer.draw(model="toy", metric="loss", x_axis="iteration", smoothing=5)

In [ ]:
drawer.draw(model="toy", metric="accuracy", x_axis="iteration", smoothing=5)

**Reading the plot:** both curves should go down (loss) or up (accuracy). If training keeps
improving while validation gets worse, the network is memorising its training events
(**overfitting**). With 200 iterations you will mostly see the steep beginning of the curve.
Real trainings run for tens of thousands of iterations.

## 4. Evaluate every checkpoint on held-out events

[`../configs/uresnet_ppn_validation.yaml`](../configs/uresnet_ppn_validation.yaml) is the same
recipe with the training parts removed. Giving `--weight-path` a **quoted pattern** makes SPINE
loop over all matching checkpoints and write one `inference_log-<N>.csv` per checkpoint. We put
them in `posthoc/`, separate from the training logs.

In [ ]:
te.sh(f"""spine -c $SPINE_TUTORIAL/configs/uresnet_ppn_validation.yaml \
          -s {TEST_FILE} -n 128 \
          --weight-path "{RUN}/weights/snapshot-[0-9]*.ckpt" \
          --log-dir {RUN}/posthoc""")

In [ ]:
rows = []
for f in sorted(glob.glob(str(RUN / "posthoc" / "inference_log-*.csv"))):
    log = pd.read_csv(f)
    rows.append({"checkpoint after N iterations": int(Path(f).stem.split("-")[-1]),
                 "mean loss": log["loss"].mean(),
                 "mean accuracy": log["accuracy"].mean() if "accuracy" in log else float("nan")})
pd.DataFrame(rows).sort_values("checkpoint after N iterations")

## 5. Use your weights in the full chain

`--module-weight uresnet_ppn=<checkpoint>` loads the official full-chain weights, then swaps
in your UResNet+PPN. After only 200 iterations it is much worse than the official one, so
compare the outputs in Part 2's notebook and see the reconstruction degrade.

In [ ]:
te.sh(f"""spine -c infer/generic/full_chain_240805.yaml \
          -s {te.fetch('generic_small.root')} \
          --module-weight uresnet_ppn={RUN}/weights/snapshot-best.ckpt \
          --output-dir {env['WORKDIR']}/out --output-suffix my_uresnet \
          --log-dir {env['WORKDIR']}/logs""")

Open `$WORKDIR/out/generic_small_my_uresnet.h5` in
[`02b_analyze_spine_output.ipynb`](../02_analysis/02b_analyze_spine_output.ipynb)
(set `full_file` to it) and compare its efficiency and purity with the official weights.

## Where next?

- **Longer and bigger:** submit the same training as a batch job with `submit.py --stage train`. See [`README.md`](README.md) §6.
- **The whole chain:** read `pipelines/protodune-sp/full_chain_260906.yaml` and dry-run it. See [`README.md`](README.md) §8.
- **Publish weights:** see [`README.md`](README.md) §7.